# 純 RL 自訓 **v1i**：v1b 熱啟動 ＋ 左右鏡像增強 ＋ roll 姿態角懲罰 ＋ 落地衝擊懲罰 ＋ 動作延遲／IMU 偏差 DR · Colab GPU（2026-10-21）

**從 v1b 熱啟動**（`INIT_FROM`，normalizer／policy／value 都還原，步數從 0 算）續訓；v1c 從頭訓 5 億步的結果是 policy 完全不抬腳、側向軸死掉（`outputs/rl_wheel/eval_rl_only_real_v1c_s1.md`、`outputs/reward_audit_wheel_v1c_s1.md`：平移 0.5 的 lin 0.04 vs v1b 2.40），判斷是 `impact` 把早期笨拙落地的探索殺掉。環境 `WheelEnv(**rw.V1I)`（spec `docs/superpowers/specs/2026-10-20-rl-wheel-v1c-design.md`；v1b 的 reward 盤點 `docs/results/Y_v1b_reward總表_2026-10-20.md`）：
- **鏡像增強**：每回合 50% 把 obs 左右翻給 policy、動作翻回物理 → policy 本身左右對稱（v1b 左平移持續側傾 7–10°、右 1–4°，三版 policy 歪的方向隨機）。
- **`roll`**：罰真實重力 y 分量平方（w 10，roll 9° ≈ −0.25／步）；原本的 `angxy` 只罰角速度，固定側傾時是 0。
- **`impact`**：罰觸地前一步足端下落速度**超過 0.3 m/s 的部分**的平方（w 3，`impact_v0` 0.3；10k 等級 0.10–0.19 免費、v1b 等級 0.43–0.71 仍罰）；原本沒有任何接觸力項。
- **DR**：IMU roll／pitch 常數偏差每回合抽 ±2°（只動 policy 的重力 obs）。**不加動作延遲 DR**：v1d（有延遲 DR）平移時膝伸到 42°、機身 0.59 m、pitch 晃 3 倍、直走超速 +0.1，v1b 沒有延遲 DR 實機 24 趟走得好（使用者 10-21 定）。
- **防「拖腿橫滑」（v1g 的改動）**：`slip` 0.075 → **0.75**，加新項 **`stuck`**（w 1.0）：側移或旋轉指令下，任一腿足端淨空 < 5 mm 連續超過 0.6 s 就罰（超過秒數、上限 1 s）；直走、站立不罰。v1f 平移時只有兩腿踏、另外兩腿全程觸地被拖著橫滑 0.5 m/s（每步只扣 0.04，比抬腳＋重心轉移便宜），直腿外張是拖腿的幾何必然；原廠平移四腿都踏、次要腿每週期原地重放 8–10 mm（L 文件）→ 不觸發 stuck。只調 slip ×10 的關卡沒過（vy 0.3 v1f 仍勝）才加 stuck。攤帳關卡：V1G 下 v1b 四腿步態每步總分要高於 v1f 拖腿步態。
- **腳要在髖下方（v1i 的改動，使用者定）**：v1h 平移時髖角從 ±46° 開到 7–19°，前腳在髖前 14–21 cm、後腳在髖後 9–17 cm，前後腳距 95–98 cm（v1b 60–75）；站高、鎖膝、拖腿都被封後，policy 用拉長支撐面讓踏步時機身不晃，髖開＋膝 65° 的髖到腳距離仍長＝繞過膝 hinge。新項 **`splay`**（w 3）罰腳相對髖在機身前後方向超過 **8 cm** 的部分（v1b 平移 ±8 cm 內、站立 ±1 cm）。關卡：V1I 下 v1b 要贏 v1h。
- **鎖姿態（v1h 的改動，使用者定「1＋2」）**：v1g 中場平移膝 50–58°（比 hinge 60° 更直、每步寧願付 knee −0.24）、機身 0.57 m（v1b 0.53），動機是 roll 懲罰在地形上逼它站直（直腿垂直剛度高）。1 直接鎖：膝 hinge 60° → **65°**（70° 會讓 v1b 自己平地扣 −0.4、樓梯 −0.6，關卡 12 列輸 2 列）、w 0.5 → **2.0**；新項 **`height`**（w 10）罰機身對地形高度超過 **0.56 m** 的部分（v1b 平移 0.53–0.56）。2 減壓：`roll` 10 → **5**。關卡加地形列（rough／boxes／stairs 第 9 級側移），v1b 要每列都贏。
- **防「用伸直腿、前後晃躲側傾」**：`angxy` 0.05 → **0.5**（罰 roll／pitch **角速度**，樓梯上的靜態俯仰不受罰）；新項 `knee`（w 0.5）罰膝比 **60°** 更直的部分（v1b 平移膝 64–80° 不罰、v1d 的 42° 罰）。
- 不動：`wgap`／`ww`／`lim`／`w_ang`／σ 0.25／`acc_leg`／`rate`／課程／推力／網路。狗端零改動，`--grav-offset 0.7 -1.5` 照舊。
- 開訓前關卡已過：`outputs/reward_audit_wheel_{10k,v1b}_v1c.md`（`impact` 10k < v1b 一半、`roll` 左 > 右）。

**流程**：第 4 格掛載雲端硬碟（會跳一次授權，按完就可以離開）→ throughput 測試選 `NUM_ENVS` → 訓練（每次 eval 存 pkl 到 `/content/rl_wheel_v1i/` **並自動複製到雲端硬碟 `MyDrive/rl_wheel_v1i/`**，人不在、runtime 斷線都不會丟）→ 一個 session 5 億步（G4、4096 環境約 50 分鐘；使用者：有必要可以多跑幾個 session）→ 下個 session 掛載後 `RESUME="auto"` 自動從雲端硬碟接最新的 step 檔，不用手動上傳。
**5 億步中場檢查**：session 1 結束把 step 檔拉回本機 `export_policy_wheel.py` → 跑 spec §6 全表；roll 左右差沒縮、或驗收表的「落地 vz」沒比 v1b 低 → 先調權重再跑 session 2；Colab 進度列的 `land_vz` 是 MJX 接觸口徑（v1b 左／右平移 0.43／0.70），本機驗收表的「落地 vz」是 CPU 淨空口徑（v1b 0.36／0.37），**兩者不能互比**，中場判讀以本機驗收表前後比為準。
**停損**：累計 2 億步時 `len < 600`、或 `lin < 0.5`、或 `wgap` 沒比 step 0 好／`gap` 中位 < 40 mm、或 `impact`、`roll` 沒比 step 0 往 0 走、或 `level` 明顯低於 v1b 同步數的曲線（崎嶇地 `impact` 量級大：樓梯 8 側移 −0.75／步、方格 8 −0.48，平地 −0.17；policy 可能逃避高等級地形）→ 停。
**讀法**：`splay` 起點（v1b）≈ 0、往 0；`height` 往 0（v1b 起點平移時 ≈ 0）、`knee` 起點（v1b 在 65° 地板下）約 −0.1 往 0；`stuck` 起點（v1b）應 ≈ 0、全程 ≈ 0（變負＝有腿被拖）；`slip` 起點約 −0.1，往 0；`knee` 往 0（v1b 起點應 ≈ 0）、`angxy` 看有沒有比 v1b 起點（約 −0.1）大很多；`lin`／`ang` 往 2.5／2.5；`roll`、`impact` 往 0；`land_vz` ＝ 每次落地平均下落速度 m/s（Σvz／Σ落地次數，不是除步數；MJX 口徑，v1b 平移 0.5 左／右 0.43／0.70，與本機驗收表的 0.36 不同口徑），往下；`mir` 應在 0.5 附近、`d` 在 1.0 附近（0–2 平均）；eval 也開鏡像／延遲／偏差，所以 eval reward 不是乾淨名目條件的數字，中場判讀以本機驗收表為準；其餘同 v1。


In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"

# 版本鎖死，不要放寬。brax 的預設 activation/分布由版本決定，而 activation
# 不匹配時 brax 載權重【不會報錯】，只會讓 policy 靜默錯亂。
# 這裡的版本 = 本機推論端 (task7/inference/) 的版本。
#
# jax<0.10：brax 0.14.2 的 ppo.train (train.py:756) 用 jax.device_put_replicated，
# 該 API 在 jax 0.10 已被移除（本機 jax 0.10.2 實測直接 AttributeError）。
# 用 jax[cuda12] 這個 extra 是為了讓 jaxlib 與 CUDA plugin 一起降到相容版本，
# 只寫 "jax<0.10" 會留下版本不合的 jaxlib/cuda plugin。
# ⚠️ 這一項無法在本機驗證：裝完務必看下一格印出的 devices 有沒有 cuda。
#    掉回 CPU 或裝不起來 → 回報，【不要】自行改 brax 版本（會動到 activation 預設值）。
!pip install -q "brax==0.14.2" "mujoco==3.10.0" "mujoco-mjx==3.10.0" "jax[cuda12]<0.10" mediapy
print("done")

In [ ]:
import jax
print("JAX", jax.__version__, "devices:", jax.devices())   # 要看到 cuda

# 版本斷言：不匹配當場停住，不要繞過。訓練跑完才發現行為對不上就白費了。
import brax, mujoco
print("brax", brax.__version__, "| mujoco", mujoco.__version__)
assert brax.__version__ == "0.14.2", (
    f"brax 版本為 {brax.__version__}，本機推論端是 0.14.2。"
    "版本不同會改變 make_ppo_networks 的預設 activation，"
    "而 activation 不匹配時 brax 載權重【不會報錯】，只會讓 policy 行為錯亂。"
    "請回到上一格重跑安裝（Colab 有時需要「執行階段 → 重新啟動工作階段」才會生效）。"
)
assert mujoco.__version__ == "3.10.0", (
    f"mujoco 版本為 {mujoco.__version__}，本機是 3.10.0。"
    "MJX 的接觸/求解器行為隨版本改變，訓練與推論不同版會讓步態對不上。"
)
# jax 0.10 移除了 device_put_replicated，而 brax 0.14.2 的 ppo.train 會用它。
# 在這裡早死，不要拖到訓練那一格編譯完才炸。
assert hasattr(jax, "device_put_replicated"), (
    f"jax {jax.__version__} 已移除 device_put_replicated，brax 0.14.2 的 ppo.train 會失敗。"
    "需要 jax<0.10。若 Colab 無法在此版本下取得 GPU 支援，請回報——"
    "換 brax 版本會改變 make_ppo_networks 的預設 activation，那會讓權重與本機推論端靜默不匹配。"
)
if not any(d.platform == "gpu" for d in jax.devices()):
    print("⚠️ 沒抓到 GPU。確認「執行階段 → 變更類型 → GPU」，"
          "以及上一格的 jax[cuda12]<0.10 是否把 CUDA 支援裝掉了。")
print("版本 OK")

In [ ]:
import os, subprocess, sys

REPO = "https://github.com/HGLLLLL/RBTDOG_SIM.git"
BRANCH = "main"
DEST = "rbtdog_sim"
if not os.path.exists(DEST):
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO, DEST], check=True)
sys.path.insert(0, f"{DEST}/task7/inference")
print("clone 到的 commit：",
      subprocess.run(["git", "-C", DEST, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)


In [ ]:
# ---- 雲端硬碟（v1i，使用者 2026-10-20 要求：按下去就能離開）：先授權掛載；每次 eval 的 step 檔與最後的 params 都自動複製到 MyDrive/rl_wheel_v1i/，續訓也從那裡自動找
from google.colab import drive
drive.mount("/content/drive")
import os
GDRIVE = "/content/drive/MyDrive/rl_wheel_v1i"; os.makedirs(GDRIVE, exist_ok=True)
print("雲端硬碟 OK →", GDRIVE, "｜已有：", sorted(os.listdir(GDRIVE)))


In [ ]:
import jax, jax.numpy as jnp, numpy as np, mujoco, time
import rl_env_wheel as rw
env = rw.WheelEnv(**rw.V1I)
print("obs", env.observation_size, "act", env.action_size, "Z_STAND", round(env.Z_STAND, 3), "scene", env.scene)
assert env.observation_size == {"state": (53,), "privileged_state": (56,)} and env.action_size == 16
# 煙霧：4 個環境零動作 20 步，不該摔
w = rw.wrap_env(env, episode_length=rw.EPISODE_LEN)
s = jax.jit(w.reset)(jax.random.split(jax.random.PRNGKey(0), 4)); step = jax.jit(w.step)
for _ in range(20):
    s = step(s, jnp.zeros((4, 16)))
assert s.obs["state"].shape == (4, 53) and s.obs["privileged_state"].shape == (4, 56)
assert float(np.asarray(s.metrics["fell"]).max()) == 0.0, "零動作就摔：出生高或地形有問題"
print("煙霧 OK；平均 reward", float(np.asarray(s.reward).mean()))


In [ ]:
# ---- throughput 測試（spec §5）：選最大且比前一檔至少快 1.3 倍的環境數（G4 實測 1024 94k／2048 145k／4096 211k）；記憶體爆就退一檔
import subprocess
bench = {}
for NUM_ENVS in (1024, 2048, 4096):
    try:
        w = rw.wrap_env(env, episode_length=rw.EPISODE_LEN)
        s = jax.jit(w.reset)(jax.random.split(jax.random.PRNGKey(0), NUM_ENVS)); step = jax.jit(w.step)
        a = jnp.zeros((NUM_ENVS, 16))
        s = step(s, a); s.obs["state"].block_until_ready()                                  # 編譯
        t0 = time.time()
        for _ in range(200):
            s = step(s, a)
        s.obs["state"].block_until_ready(); dt = time.time() - t0
        bench[NUM_ENVS] = NUM_ENVS * 200 / dt
        mem = subprocess.run(["nvidia-smi", "--query-gpu=memory.used,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        print(f"{NUM_ENVS} envs: {bench[NUM_ENVS]:,.0f} steps/s  GPU {mem}")
    except Exception as e:                                                          # 多半是 OOM
        print(f"{NUM_ENVS} envs: 失敗 {type(e).__name__}: {str(e)[:120]}"); break
s = w = step = a = None; del s, w, step, a                                    # 釋放 bench 的環境狀態，免得佔住 GPU 記憶體
NUM_ENVS = 1024
for n in (2048, 4096):                                                          # 遇到第一檔不合 1.3 倍就停，不跳級
    if n in bench and bench[n] >= 1.3 * bench[n // 2]:
        NUM_ENVS = n
    else:
        break
print("→ NUM_ENVS =", NUM_ENVS, "| 10 億步預估", f"{1e9 / bench[NUM_ENVS] / 3600:.1f} 小時（下限：純 step 速率，不含 PPO 更新與 eval）")


In [ ]:
import functools, time
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks
from brax.io import model
DRIVE = "/content/rl_wheel_v1i"; import os, re, glob; os.makedirs(DRIVE, exist_ok=True)   # 本機暫存；每個 step 檔另外複製到 GDRIVE（雲端硬碟，上一格掛載），斷線也不丟

# 續訓：把上一輪下載的 rl_wheel_v1i_step*.pkl 上傳到 /content/rl_wheel_v1i/，RESUME 指到「檔名步數最大」的那個（每次 eval 存一個；(normalizer, policy, value) 三元組）。
# 不要指 rl_wheel_v1i_params.pkl：它檔名沒有步數。None ＝ 從頭。
# brax 的 restore_params 不還原步數，STEP_OFFSET 從 RESUME 檔名解析累計步數，存檔名與進度列印都用累計步數；停損線「2 億步」是累計步數。
# TIMESTEPS 是「這一輪再跑幾步」；一個 session 5 億（v1i 從 v1b 熱啟動，每 session 5 億，可多跑）。
# RESUME = "auto"：自動在 DRIVE 裡找步數最大的 rl_wheel_v1i_step*.pkl（把上一輪下載的檔上傳進去就會接著訓）；None ＝ 強制從頭；也可直接給路徑。
RESUME = "auto"
if RESUME == "auto":
    _cands = sorted(glob.glob(f"{DRIVE}/rl_wheel_v1i_step*.pkl") + glob.glob(f"{GDRIVE}/rl_wheel_v1i_step*.pkl"), key=lambda p_: int(re.search(r"step(\d+)", p_).group(1)))
    RESUME = _cands[-1] if _cands else None
print("RESUME =", RESUME if RESUME else "None（從頭訓）")
_restore = None if RESUME is None else model.load_params(RESUME)
INIT_FROM = "rbtdog_sim/task7/weights/rl_wheel_v1b_params.pkl"     # v1d：沒有 v1d 的 step 檔時從 v1b 熱啟動（normalizer＋policy＋value 都還原；累計步數從 0 算）
if _restore is None and INIT_FROM:
    _restore = model.load_params(INIT_FROM); print("熱啟動自", INIT_FROM)
def _step_offset(path):
    """續訓：從 RESUME 檔名解析累計步數；檔名沒有 step<N> 就報錯（不要靜默從 0 算）。"""
    m_ = re.search(r"step(\d+)", path)
    if m_ is None:
        raise ValueError("RESUME 要指到 rl_wheel_v1i_step<N>.pkl，不是 rl_wheel_v1i_params.pkl")
    return int(m_.group(1))
STEP_OFFSET = 0 if RESUME is None else _step_offset(RESUME)
TIMESTEPS = 500_000_000

env = rw.WheelEnv(**rw.V1I)
eval_env = rw.WheelEnv(noise=False, push=False, **rw.V1I)
network_factory = functools.partial(ppo_networks.make_ppo_networks,
                                    policy_hidden_layer_sizes=(512, 256, 128), value_hidden_layer_sizes=(512, 256, 128),
                                    policy_obs_key="state", value_obs_key="privileged_state")   # critic 特權 obs 56
_t0 = time.time(); rewards = []

def _save_ckpt(step, make_policy, params):
    p_ = f"{DRIVE}/rl_wheel_v1i_step{STEP_OFFSET + step:012d}.pkl"; model.save_params(p_, params)
    import shutil; shutil.copy(p_, GDRIVE)                     # 自動上傳雲端硬碟（每次 eval 一份，runtime 斷線也不丟）

def progress(step, metrics):
    r = float(metrics.get("eval/episode_reward", 0.0)); total = STEP_OFFSET + step; rewards.append((total, r))   # 累計步數
    L = float(metrics.get("eval/avg_episode_length", 1.0)) or 1.0
    ps = lambda k: float(metrics.get(f"eval/episode_{k}", 0.0)) / L
    ev = lambda k: float(metrics.get(f"eval/episode_{k}", 0.0))                  # 回合總和（不除步數）
    land_vz = ev('impact_vz') / max(ev('td_n'), 1e-9)                              # 每次落地平均下落速度 m/s ＝ Σvz／Σ落地次數（除步數會被無落地的步稀釋）
    el = time.time() - _t0; rate = step / max(el, 1e-9)                         # 本 session 速率（ETA 用）
    print(f"step {total:>11,} R {r:7.2f} | lin {ps('lin'):.2f}/2.5 ang {ps('ang'):.2f}/2.5 small {ps('lin_small'):.2f} | "
          f"ww {ps('ww'):+.3f} wgap {ps('wgap'):+.3f} gap {ps('gap_min')*1000:.0f}mm slip {ps('slip'):+.3f} dev {ps('dev'):+.3f} rate {ps('rate'):+.3f} lim {ps('lim'):+.3f} sat {ps('sat_n'):.2f} | roll {ps('roll'):+.3f} impact {ps('impact'):+.3f} knee {ps('knee'):+.3f} angxy {ps('angxy'):+.3f} stuck {ps('stuck'):+.3f} slip {ps('slip'):+.3f} height {ps('height'):+.3f} splay {ps('splay'):+.3f} land_vz {land_vz:.2f} mir {ps('mir'):.2f} d {ps('delay'):.1f} | "
          f"vxerr {ps('vx_err'):.2f} vyerr {ps('vy_err'):.2f} wzerr {ps('wz_err'):.2f} | level {ps('level'):.1f} fell {ps('fell') * L:.2f} len {L:.0f} | "
          f"{el:.0f}s → {TIMESTEPS / max(rate, 1) / 60:.0f} 分")

train_fn = functools.partial(
    ppo.train, num_timesteps=TIMESTEPS, num_evals=20, episode_length=rw.EPISODE_LEN,
    num_envs=NUM_ENVS, batch_size=256 * NUM_ENVS // 2048, num_minibatches=32, unroll_length=20,
    num_updates_per_batch=4, learning_rate=3e-4, entropy_cost=1e-2,
    discounting=0.99, normalize_observations=True,
    learning_rate_schedule="ADAPTIVE_KL", desired_kl=0.01, learning_rate_schedule_min_lr=1e-5, learning_rate_schedule_max_lr=1e-3,
    network_factory=network_factory, randomization_fn=rw.make_domain_randomize(env), wrap_env_fn=rw.wrap_env,
    seed=0, eval_env=eval_env, policy_params_fn=_save_ckpt, restore_params=_restore)
make_inference_fn, params, _ = train_fn(environment=env, progress_fn=progress)
print("training done")


In [ ]:
import matplotlib.pyplot as plt
plt.plot([s for s, _ in rewards], [r for _, r in rewards], marker="o"); plt.xlabel("env steps"); plt.ylabel("eval reward"); plt.grid(True); plt.show()


In [ ]:
model.save_params("rl_wheel_v1i_params.pkl", params)
import shutil; shutil.copy("rl_wheel_v1i_params.pkl", f"{DRIVE}/rl_wheel_v1i_params.pkl"); shutil.copy("rl_wheel_v1i_params.pkl", f"{GDRIVE}/rl_wheel_v1i_params.pkl")
print("已存 rl_wheel_v1i_params.pkl（/content 與 /content/rl_wheel_v1i/）→ ✅ 已自動上傳雲端硬碟 MyDrive/rl_wheel_v1i/（step 檔每次 eval 也都有副本），回來從那裡拿 → 放 task7/weights/；本機：export_policy_wheel.py → rl_only_infer.py --npz … --self-collision")
